In [6]:
# Minimal pose stream using the simple startMission overload (no ClientPool)
import json, time, os, shutil
from malmo.MalmoPython import AgentHost, MissionSpec, MissionRecordSpec

MISSION = """<?xml version="1.0" encoding="UTF-8" standalone="no" ?>
<Mission xmlns="http://ProjectMalmo.microsoft.com" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">

  <About>
    <Summary>Extracting camera pose and frames for later SLAM</Summary>
  </About>

  <ServerSection>
    <ServerInitialConditions>
            <Time>
                <StartTime>3000</StartTime>
                <AllowPassageOfTime>false</AllowPassageOfTime>
            </Time>
            <Weather>clear</Weather>
            <AllowSpawning>false</AllowSpawning>
    </ServerInitialConditions>
    
    <ServerHandlers>
      <FlatWorldGenerator generatorString="3;1*minecraft:dirt,1*minecraft:grass;2;village" forceReset="true"/>
      
      <DrawingDecorator>
        <!-- Side walls -->
        <DrawCuboid x1="-2" y1="2" z1="-1" x2="-2" y2="5" z2="10" type="stone"/>
        <DrawCuboid x1="2"  y1="2" z1="-1" x2="2"  y2="5" z2="7" type="stone"/>
        
        <!-- Back wall -->
        <DrawCuboid x1="-2" y1="2" z1="-2" x2="2" y2="5" z2="-2" type="stone"/>
        
        <!-- Left turn -->
        <DrawCuboid x1="-2" y1="2" z1="11" x2="7" y2="5" z2="11" type="stone"/>
        <DrawCuboid x1="2"  y1="2" z1="8" x2="7"  y2="5" z2="8" type="stone"/>

        <!-- Left turn wall -->
        <DrawCuboid x1="8" y1="2" z1="8" x2="8" y2="5" z2="11" type="stone"/>
        
        <!-- Ceiling -->
        <DrawCuboid x1="-2" y1="5" z1="-1" x2="2"  y2="5" z2="11" type="stone"/>
        <DrawCuboid x1="2" y1="5" z1="8" x2="8"  y2="5" z2="11" type="stone"/>

        <!-- Light-->
        <DrawBlock x="-1"  y="2" z="-1" type="torch"/>
        <DrawBlock x="-1"  y="2" z="2" type="torch"/>
        <DrawBlock x="-1"  y="2" z="5" type="torch"/>
        <DrawBlock x="-1" y="2" z="7" type="torch"/>
        <DrawBlock x="-1" y="2" z="10" type="torch"/>
        <DrawBlock x="2" y="2" z="10" type="torch"/>
        <DrawBlock x="5" y="2" z="10" type="torch"/>
      </DrawingDecorator>
      
      <ServerQuitFromTimeUp description="" timeLimitMs="5000"/>
    </ServerHandlers>
    
  </ServerSection>

  <AgentSection mode="Survival">
    <Name>Amidouguis</Name>
    
    <AgentStart>
      <Placement pitch="0" x="0.0" y="2.0" yaw="0" z="0.0"/>
    </AgentStart>
    
    <AgentHandlers>
      <ContinuousMovementCommands turnSpeedDegs="180"/>
      
      <ObservationFromFullStats/>
      
      <VideoProducer want_depth="true">
        <Width>640</Width>
        <Height>360</Height>
      </VideoProducer>
    </AgentHandlers>
    
  </AgentSection>

</Mission>
"""

# Setting up folders
root_dir = "./data"
images_dir = os.path.join(root_dir, "images")
shutil.rmtree(root_dir, ignore_errors=True)
os.makedirs(images_dir, exist_ok=True)
pose_path = os.path.join(root_dir, "pose.txt")

# Instantiating agent and mission
ah = AgentHost()
ms = MissionSpec(MISSION, True)
mr = MissionRecordSpec()

print("Initializing mission.\n")
ah.startMission(ms, mr)

# Mission start error handling
t0 = time.time()
while True:
    ws = ah.getWorldState()
    
    if ws.has_mission_begun:
        break
    
    if any(ws.errors):
        for e in ws.errors: 
            print("Error:", e.text)
        raise RuntimeError("Mission failed to start due to above errors.")
    
    if time.time() - t0 > 10:
        raise RuntimeError("Mission failed to start due to timeout.")
    
    time.sleep(0.1)
    
# Data extraction loop
frame_idx = 1

with open(pose_path, "w", encoding="utf-8") as pose_file:
    while ws.is_mission_running:
        ws = ah.getWorldState()

        last_pose = False
        last_frame = False
        
        if ws.observations:
            obs = json.loads(ws.observations[-1].text)
            x, y, z = float(obs.get("XPos")), float(obs.get("YPos")), float(obs.get("ZPos"))
            yaw, pitch = float(obs.get("Yaw")), float(obs.get("Pitch"))
            last_pose = True
            
        if ws.video_frames:
            frame = ws.video_frames[0]
            
            image_filename = "{:05d}.ppm".format(frame_idx)
            image_path = os.path.join(images_dir, image_filename)
            header = "P6 {} {} 255\n".format(frame.width, frame.height)
            with open(image_path, "wb") as f:
                f.write(header.encode("ascii"))
                f.write(bytes(frame.pixels))
            
            frame_idx += 1
            last_frame = True
                
        if last_pose and last_frame:
            pose_file.write("{:.4f},{:.4f},{:.4f},{:.4f},{:.4f},{}\n".format(x, y, z, yaw, pitch, image_path))
            pose_file.flush()

        time.sleep(0.05)

Initializing mission.

